# Occupancy prediction demo

This notebook starts from a cleaned 81-column `pandas.DataFrame` with two-level columns `(room, sensor)`. It does not include any database export, raw-data cleaning, interpolation, or normalization code.

The demo trains and compares eight models:

1. CNN
2. CNN + position coding
3. LSTM
4. LSTM + position coding
5. GNN + CNN
6. GNN + CNN + position coding
7. GNN + LSTM
8. GNN + LSTM + position coding


In [1]:
import torch
import pandas as pd

import occupancy_models as om

om.set_seed(42)
torch.set_num_threads(2)

device = om.get_device()
print(f"Using device: {device}")

Using device: cuda


d:\App\conda\envs\MAIHOME\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load or provide the 81-column DataFrame

Replace the synthetic DataFrame below with your real cleaned DataFrame. The real DataFrame should already have the same 81 columns and should use a two-level MultiIndex: `(room_name, sensor_name)`.


In [2]:
# Replace this synthetic DataFrame with your own cleaned 81-column DataFrame.
# Example:
# df = pd.read_parquet("your_clean_81_column_dataframe.parquet")

USE_SYNTHETIC_DATA = True

if USE_SYNTHETIC_DATA:
    df = om.make_synthetic_dataframe(n_steps=180, seed=42)
else:
    raise ValueError("Assign your cleaned 81-column DataFrame to `df` here.")

om.validate_input_dataframe(df, expected_columns=om.DEFAULT_COLUMNS, target_rooms=om.TARGET_ROOMS)
print(df.shape)
df.head()

(180, 81)


Badkamer                                      \
                    Pirstatus    Pirsum MotorPosition Temperature   
2025-01-01 00:00:00       0.0  0.236004      0.301192    0.233441   
2025-01-01 00:10:00       0.0  0.053466      0.540828    0.300997   
2025-01-01 00:20:00       0.0  0.111058      0.356518    0.359571   
2025-01-01 00:30:00       0.0  0.295471      0.494132    0.414247   
2025-01-01 00:40:00       1.0  1.000000      0.511644    0.571678   

                                               Eetkamer                       \
                    TemperatureSet Lightlevel Pirstatus    Pirsum Lightlevel   
2025-01-01 00:00:00       0.263683   0.351006       1.0  0.958193   0.545768   
2025-01-01 00:10:00       0.226611   0.409400       0.0  0.210262   0.386913   
2025-01-01 00:20:00       0.335843   0.305507       1.0  0.981523   0.582962   
2025-01-01 00:30:00       0.431643   0.373741       1.0  1.000000   0.577591   
2025-01-01 00:40:00       0.662298   0.502312       1.0  0.954727   0.574393   

                    Hal beneden  ...   slaapkamer 3                      \
                      AccMotion  ... TemperatureSet         X         Y   
2025-01-01 00:00:00    0.448346  ...       0.446295  0.234059  0.473739   
2025-01-01 00:10:00    0.404821  ...       0.457602  0.362662  0.453793   
2025-01-01 00:20:00    0.380044  ...       0.406248  0.293679  0.520210   
2025-01-01 00:30:00    0.388603  ...       0.630491  0.434978  0.565247   
2025-01-01 00:40:00    0.520995  ...       0.377973  0.327266  0.420742   

                              watermeter                  Time            \
                            Z   Humidity Temperature  hour_sin  hour_cos   
2025-01-01 00:00:00  0.339252   0.504516    0.383616  0.000000  1.000000   
2025-01-01 00:10:00  0.295053   0.634952    0.451599  0.043619  0.999048   
2025-01-01 00:20:00  0.291426   0.403508    0.432437  0.087156  0.996195   
2025-01-01 00:30:00  0.547286   0.361198    0.389160  0.130526  0.991445   
2025-01-01 00:40:00  0.343253   0.707753    0.521760  0.173648  0.984808   

                                         
                      dow_sin   dow_cos  
2025-01-01 00:00:00  0.974928 -0.222521  
2025-01-01 00:10:00  0.974928 -0.222521  
2025-01-01 00:20:00  0.974928 -0.222521  
2025-01-01 00:30:00  0.974928 -0.222521  
2025-01-01 00:40:00  0.974928 -0.222521  

[5 rows x 81 columns]

## Experiment configuration

The values below are small enough for a public demo. For the full paper experiment, use the original settings: `INPUT_LEN = 72`, `PRED_LEN = 36`, `NUM_WINDOWS = 5`, and a larger number of training epochs.


In [17]:
INPUT_LEN = 72
PRED_LEN = 36
MAX_SIZE = 81
TARGET_ROOMS = om.TARGET_ROOMS

# Demo settings. Increase these for the paper-scale run.
NUM_WINDOWS = 2
START_WINDOW = 2
EPOCHS = 1
PATIENCE = 1
MIN_EPOCHS = 0
BATCH_SIZE = 32

# Smaller hidden dimensions make the demo fast on CPU.
# Increase these for the paper-scale run.
GNN_HIDDEN = 32
LSTM_HIDDEN = 128

## Build model inputs

The same cleaned DataFrame is converted into the required tensor format for each model family. Position coding is based on shortest-path distance over the room graph.


In [18]:
model_inputs = om.build_all_model_inputs(
    df,
    input_len=INPUT_LEN,
    pred_len=PRED_LEN,
    max_size=MAX_SIZE,
    target_rooms=TARGET_ROOMS,
    show_progress=True,
)

for model_name, spec in model_inputs.items():
    x = spec["x"]
    y = spec["y"]
    print(f"{model_name:32s} X={x.shape} y={y.shape}")

Building graph windows: 100%|██████████| 73/73 [00:00<00:00, 73057.55it/s]

CNN                              X=(73, 72, 81) y=(73, 5)
CNN + position coding            X=(73, 72, 6, 81) y=(73, 5)
LSTM                             X=(73, 72, 81) y=(73, 5)
LSTM + position coding           X=(73, 72, 486) y=(73, 5)
GNN + CNN                        X=(73, 72, 14, 24) y=(73, 5)
GNN + CNN + position coding      X=(73, 72, 14, 29) y=(73, 5)
GNN + LSTM                       X=(73, 72, 14, 24) y=(73, 5)
GNN + LSTM + position coding     X=(73, 72, 14, 29) y=(73, 5)


## Train and evaluate all eight models

In [19]:
results = {}

for model_name, spec in model_inputs.items():
    print()
    print(f"===== {model_name} =====")
    model_factory = om.create_model_factory(
        model_name=model_name,
        x_shape=spec["x"].shape,
        output_dim=len(TARGET_ROOMS),
        graph_info=spec["graph_info"],
        gnn_hidden=GNN_HIDDEN,
        lstm_hidden=LSTM_HIDDEN,
    )
    results[model_name] = om.run_expanding_window_experiment(
        x=spec["x"],
        y=spec["y"],
        model_factory=model_factory,
        target_rooms=TARGET_ROOMS,
        num_windows=NUM_WINDOWS,
        start_window=START_WINDOW,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        patience=PATIENCE,
        min_epochs=MIN_EPOCHS,
        device=device,
        verbose=False,
    )


===== CNN =====

===== CNN + position coding =====

===== LSTM =====

===== LSTM + position coding =====

===== GNN + CNN =====

===== GNN + CNN + position coding =====

===== GNN + LSTM =====

===== GNN + LSTM + position coding =====


## Summary across models

In [20]:
summary = om.summarize_results(results)
summary

,model,accuracy,recall,precision,f1,auc
0,CNN,1.0,1.0,1.0,1.0,NaN
1,CNN + position coding,1.0,1.0,1.0,1.0,NaN
2,LSTM,0.6,0.6,0.6,0.6,NaN
3,LSTM + position coding,0.2,0.2,0.2,0.2,NaN
4,GNN + CNN,1.0,1.0,1.0,1.0,NaN
5,GNN + CNN + position coding,1.0,1.0,1.0,1.0,NaN
6,GNN + LSTM,0.4,0.4,0.4,0.4,NaN
7,GNN + LSTM + position coding,0.8,0.8,0.8,0.8,NaN


## Per-room metrics for one selected model

In [22]:
selected_model = summary.iloc[0]["model"]
print(f"Selected model: {selected_model}")
results[selected_model]["average_metrics"]

Selected model: CNN


,room,accuracy,recall,precision,f1,auc
0,Badkamer,1.0,1.0,1.0,1.0,NaN
1,Eetkamer,1.0,1.0,1.0,1.0,NaN
2,Hal boven,1.0,1.0,1.0,1.0,NaN
3,Keuken,1.0,1.0,1.0,1.0,NaN
4,Living,1.0,1.0,1.0,1.0,NaN


In [23]:
results[selected_model]["per_window_metrics"].head(20)

,window,room,accuracy,recall,precision,f1,auc
0,2,Badkamer,1.0,1.0,1.0,1.0,NaN
1,2,Eetkamer,1.0,1.0,1.0,1.0,NaN
2,2,Keuken,1.0,1.0,1.0,1.0,NaN
3,2,Living,1.0,1.0,1.0,1.0,NaN
4,2,Hal boven,1.0,1.0,1.0,1.0,NaN
